# 03 - Event Alignment & Dashboard Logic

The dashboard (`dashboard/backend` + `dashboard/frontend`) is a thin presentation
layer over exactly the artifacts this pipeline produces: `data/processed/*.csv`,
`outputs/logs/trace_summary.csv`, and `outputs/reports/latest_run.json`. This
notebook demonstrates the two pieces of "dashboard logic" that aren't just a CSV
read: aligning the detected change point with curated real-world events, and
generating the analyst narrative surfaced in the dashboard's Analyst Note card.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.ai.narrative import generate_narrative
from src.config import PATHS
from src.insights.event_alignment import build_alignment_report

df = pd.read_csv(PATHS.processed_prices_csv, parse_dates=["Date"])
events = pd.read_csv(PATHS.processed_events_csv, parse_dates=["Start Date"])
summary = pd.read_csv(PATHS.trace_summary_csv, index_col=0)

tau_index = int(float(summary.loc["tau", "mean"]))
change_date = df.iloc[min(tau_index, len(df) - 1)]["Date"]
change_date

In [ ]:
alignment = build_alignment_report(df, change_date, events)
alignment.to_dict()

In [ ]:
report_for_narrative = {
    "change_point": {"change_date": str(change_date.date())},
    "diagnostics": {"reliable": True, "warnings": []},
    "event_alignment": alignment.to_dict(),
}
narrative = generate_narrative(report_for_narrative)
print(f"[{narrative['source']}]", narrative["text"])

## How this maps to the API / frontend

* `GET /api/prices`, `/api/events`, `/api/stats`, `/api/change-point` - direct reads
  of the processed CSVs / trace summary above (`dashboard/backend/data_loader.py`).
* `GET /api/insights` - returns `outputs/reports/latest_run.json`, which is exactly
  this notebook's `alignment` + `narrative` output, produced automatically by
  `pipelines/run_pipeline.py` on every run.
* The React `AnalystInsights` component (`dashboard/frontend/src/App.js`) renders
  `narrative.text` and flags the card if `diagnostics.reliable` is `false`.